In [ ]:
# import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    roc_curve
)

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)

from sklearn.pipeline import make_pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import StandardScaler

import joblib

In [ ]:
# load cleaned dataset

df = pd.read_csv("cleaned_data.csv")

print(df.shape)

df.head()

In [ ]:
# Prepare features and target

X_raw = df.drop(columns=["Survived", "PassengerId", "Name", "Ticket"])
y_clf = df["Survived"]

X_encoded = pd.get_dummies(X_raw, drop_first=True)
print("New Feature Shape:", X_encoded.shape)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load cleaned Titanic dataset
df = pd.read_csv("cleaned_data.csv")

# Features and target
X = df.drop("Survived", axis=1)
y_train_target = df["Survived"]

# Convert categorical columns to numeric
X = pd.get_dummies(X, drop_first=True)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_train_target,
    test_size=0.2,
    random_state=42,
    stratify=y_train_target
)


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
# Train Decision Tree (Default)

dt_model = DecisionTreeClassifier(random_state=42)

dt_model.fit(X_train_scaled, y_train)

In [ ]:
# Training and Testing Accuracy

train_pred = dt_model.predict(X_train_scaled)
test_pred = dt_model.predict(X_test_scaled)

train_accuracy = accuracy_score(y_train, train_pred)
test_accuracy = accuracy_score(y_test, test_pred)

print("Training Accuracy :", train_accuracy)
print("Testing Accuracy :", test_accuracy)

In [ ]:
# controlled decision tree

controlled_tree = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    random_state=42
)

controlled_tree.fit(X_train_scaled, y_train)

In [ ]:
# Training and testing accuracy

controlled_train = controlled_tree.predict(X_train_scaled)
controlled_test = controlled_tree.predict(X_test_scaled)

controlled_train_acc = accuracy_score(y_train, controlled_train)
controlled_test_acc = accuracy_score(y_test, controlled_test)

print("Controlled Tree Training Accuracy :", controlled_train_acc)
print("Controlled Tree Testing Accuracy :", controlled_test_acc)

In [ ]:
# Gini decision tree

gini_tree = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    random_state=42
)

gini_tree.fit(X_train_scaled, y_train)

gini_accuracy = accuracy_score(
    y_test,
    gini_tree.predict(X_test_scaled)
)


# Entropy decision tree

entropy_tree = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=5,
    random_state=42
)

entropy_tree.fit(X_train_scaled, y_train)

entropy_accuracy = accuracy_score(
    y_test,
    entropy_tree.predict(X_test_scaled)
)

print("Gini Accuracy :", gini_accuracy)
print("Entropy Accuracy :", entropy_accuracy)

In [ ]:
# train random forest model

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf_model.fit(X_train_scaled, y_train)

In [ ]:
# make predictions

rf_train_pred = rf_model.predict(X_train_scaled)
rf_test_pred = rf_model.predict(X_test_scaled)

# calculate accuracy

rf_train_acc = accuracy_score(y_train, rf_train_pred)
rf_test_acc = accuracy_score(y_test, rf_test_pred)

# calculate auc

rf_prob = rf_model.predict_proba(X_test_scaled)[:, 1]

rf_auc = roc_auc_score(y_test, rf_prob)

print("training accuracy :", rf_train_acc)
print("testing accuracy :", rf_test_acc)
print("roc auc :", rf_auc)

In [ ]:
# get feature importance

importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf_model.feature_importances_
})

# sort values

importance = importance.sort_values(
    by="importance",
    ascending=False
)

print(importance.head(5))

In [ ]:
# train gradient boosting model

gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb_model.fit(X_train_scaled, y_train)

In [ ]:
# make predictions

gb_train_pred = gb_model.predict(X_train_scaled)
gb_test_pred = gb_model.predict(X_test_scaled)

# calculate accuracy

gb_train_acc = accuracy_score(y_train, gb_train_pred)
gb_test_acc = accuracy_score(y_test, gb_test_pred)

# calculate auc

gb_prob = gb_model.predict_proba(X_test_scaled)[:, 1]

gb_auc = roc_auc_score(y_test, gb_prob)

print("training accuracy :", gb_train_acc)
print("testing accuracy :", gb_test_acc)
print("roc auc :", gb_auc)

In [ ]:
# make predictions

gb_train_pred = gb_model.predict(X_train_scaled)
gb_test_pred = gb_model.predict(X_test_scaled)

# calculate accuracy

gb_train_acc = accuracy_score(y_train, gb_train_pred)
gb_test_acc = accuracy_score(y_test, gb_test_pred)

# calculate auc

gb_prob = gb_model.predict_proba(X_test_scaled)[:, 1]

gb_auc = roc_auc_score(y_test, gb_prob)

print("training accuracy :", gb_train_acc)
print("testing accuracy :", gb_test_acc)
print("roc auc :", gb_auc)

In [ ]:
# find the 5 least important features

low_features = importance.sort_values(
    by="importance",
    ascending=True
).head(5)

print(low_features)

In [ ]:
# remove the least important features

remove_columns = low_features["feature"].tolist()

X_train_reduced = pd.DataFrame(
    X_train_scaled,
    columns=X.columns
).drop(columns=remove_columns)

X_test_reduced = pd.DataFrame(
    X_test_scaled,
    columns=X.columns
).drop(columns=remove_columns)
print(remove_columns)

In [ ]:
# train random forest using reduced features

rf_reduced = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf_reduced.fit(X_train_reduced, y_train)

reduced_pred = rf_reduced.predict(X_test_reduced)
reduced_prob = rf_reduced.predict_proba(X_test_reduced)[:, 1]

reduced_auc = roc_auc_score(y_test, reduced_prob)

print("full model auc :", rf_auc)
print("reduced model auc :", reduced_auc)

In [ ]:
# compare both random forest models

comparison = pd.DataFrame({
    "model": ["full model", "reduced model"],
    "auc": [rf_auc, reduced_auc]
})

print(comparison)

In [ ]:
# import required libraries

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

In [ ]:
# create models

log_model = LogisticRegression(
    max_iter=5000,
    class_weight="balanced",
    random_state=42
)

tree_model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    random_state=42
)

forest_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

boost_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

In [ ]:
# create stratified folds

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
# evaluate all models

models = {
    "logistic regression": log_model,
    "decision tree": tree_model,
    "random forest": forest_model,
    "gradient boosting": boost_model
}

for name, model in models.items():

    scores = cross_val_score(
        model,
        X,
        y_clf,
        cv=cv,
        scoring="roc_auc"
    )

    print(name)
    print("mean auc :", scores.mean())
    print("std auc :", scores.std())
    print()

In [ ]:
# import required libraries

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
import joblib

In [ ]:
# create pipeline

pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

In [ ]:
# create parameter grid

param_grid = {
    "randomforestclassifier__n_estimators": [50, 100, 200],
    "randomforestclassifier__max_depth": [5, 10, None],
    "randomforestclassifier__min_samples_leaf": [1, 5]
}

In [ ]:
# run grid search
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    ),
    scoring="roc_auc",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
print(f"Best Params: {grid_search.best_params_}")
print(f"Best Score: {grid_search.best_score_:.4f}")

# Task 7: Manual Learning Curve
best_pipeline = grid_search.best_estimator_
for f in [0.2, 0.4, 0.6, 0.8, 1.0]:
    num = int(f * len(X_train))
    best_pipeline.fit(X_train.iloc[:num], y_train.iloc[:num])
    
    tr_auc = roc_auc_score(y_train.iloc[:num], best_pipeline.predict_proba(X_train.iloc[:num])[:, 1])
    ts_auc = roc_auc_score(y_test, best_pipeline.predict_proba(X_test)[:, 1])
    print(f"Fraction: {f} | Train AUC: {tr_auc:.4f} | Test AUC: {ts_auc:.4f}")

In [ ]:
# display best result

print("best parameters")
print(grid_search.best_params_)

print()

print("best auc")
print(grid_search.best_score_)

In [ ]:
# create manual learning curve

fractions = [0.2, 0.4, 0.6, 0.8, 1.0]

results = []

best_pipeline = grid_search.best_estimator_

for f in fractions:

    size = int(f * len(X_train))

    X_subset = X_train.iloc[:size]
    y_subset = y_train.iloc[:size]

    best_pipeline.fit(X_subset, y_subset)

    train_prob = best_pipeline.predict_proba(X_subset)[:, 1]
    test_prob = best_pipeline.predict_proba(X_test)[:, 1]

    train_auc = roc_auc_score(y_subset, train_prob)
    test_auc = roc_auc_score(y_test, test_prob)

    results.append([f, train_auc, test_auc])

In [ ]:
# save the best model

joblib.dump(best_pipeline, "best_model.pkl")

print("model saved successfully")

In [ ]:
# load the saved model

loaded_model = joblib.load("best_model.pkl")

print("model loaded successfully")

In [ ]:
# make predictions using the loaded model

sample_rows = X_test.iloc[:2]

predictions = loaded_model.predict(sample_rows)

print(predictions)

In [ ]:
# fit logistic regression model

log_model.fit(X_train, y_train)

In [ ]:
# train logistic regression model

log_model.fit(X_train, y_train)

In [ ]:
# calculate decision tree auc

tree_auc = roc_auc_score(
    y_test,
    controlled_tree.predict_proba(X_test_scaled)[:, 1]
)

print("decision tree auc :", tree_auc)

In [ ]:
# calculate logistic regression auc

log_prob = log_model.predict_proba(X_test)[:, 1]

log_auc = roc_auc_score(y_test, log_prob)

print("logistic regression auc :", log_auc)

In [ ]:
# create summary table

summary = pd.DataFrame({
    "model": [
        "logistic regression",
        "decision tree",
        "random forest",
        "gradient boosting"
    ],
    "test auc": [
        auc_score,
        roc_auc_score(
            y_test,
            controlled_tree.predict_proba(X_test_scaled)[:, 1]
        ),
        rf_auc,
        gb_auc
    ]
})

print(summary)

In [ ]:
# save the best model

import joblib

best_pipeline = grid_search.best_estimator_

joblib.dump(best_pipeline, "best_model.pkl")

print("best model saved successfully")
# load the saved model

loaded_model = joblib.load("best_model.pkl")

print("model loaded successfully")

In [ ]:
# create two sample test rows

sample_data = X_test.iloc[:2]

predictions = loaded_model.predict(sample_data)

print("predictions:")
print(predictions)

In [ ]:
 #Model Serialization & Verification

# 1. Save the best pipeline model
joblib.dump(best_pipeline, 'best_model.pkl')

# 2. Reload the saved model from disk
loaded_model = joblib.load('best_model.pkl')

# 3. Create sample input from test data
sample_data = X_test.iloc[0:2]

# 4. Generate predictions using loaded model
preds = loaded_model.predict(sample_data)

# 5. Output predictions to verify success
print(f"Model Verification Passed! Predictions: {preds}")